In [ ]:
# ============================================================
# Cell 1 — Colab Setup
# Clone repo into runtime and verify structure
# ============================================================

import os

REPO_URL    = "https://github.com/manuelarceaguirre/capstone-quantum.git"
BRANCH      = "David"  # change to your branch name
REPO_ROOT   = "/content/capstone-quantum"

%cd /content

# remove stale runtime copy if it exists
!rm -rf capstone-quantum

# clone and checkout branch
!git clone {REPO_URL}
%cd /content/capstone-quantum
!git checkout {BRANCH}

# verify expected directories exist
for d in ["data/processed", "notebooks", "results", "src"]:
    path = os.path.join(REPO_ROOT, d)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  {d:<25} {status}")

# create results directory if it doesn't exist yet
os.makedirs(os.path.join(REPO_ROOT, "results"), exist_ok=True)

print("\nSetup complete.")

/content
Cloning into 'capstone-quantum'...
remote: Enumerating objects: 324, done.
remote: Counting objects: 100% (324/324), done.
remote: Compressing objects: 100% (238/238), done.
remote: Total 324 (delta 151), reused 246 (delta 84), pack-reused 0 (from 0)
Receiving objects: 100% (324/324), 14.40 MiB | 21.04 MiB/s, done.
Resolving deltas: 100% (151/151), done.
/content/capstone-quantum
  data/processed            ✓
  notebooks                 ✓
  results                   ✓
  src                       ✓

Setup complete.


In [5]:
# ============================================================
# Cell 2 — Imports
# ============================================================

import os
import sys
import json
import pickle
import time
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    median_absolute_error,
)

warnings.filterwarnings("ignore")

# add repo root to path so src/ is importable
sys.path.insert(0, REPO_ROOT)

print("Imports complete.")

Imports complete.


In [6]:
# ============================================================
# Cell 3 — Configuration
# ============================================================

# --- Paths ---
DATA_DIR    = Path(REPO_ROOT) / "data" / "processed"
RESULTS_DIR = Path(REPO_ROOT) / "results"

# --- Window Parameters ---
WINDOW_MONTHS = 120   # training window length
TEST_MONTHS   = 1     # test window length
STEP_MONTHS   = 1     # fold step size

# --- Evaluation Parameters ---
HORIZON_LIST  = [1, 3, 6]
TARGET_LIST   = ["INDPRO", "PAYEMS", "CPIAUCSL", "SP500"]
ACTIVE_TRACKS = ["A", "B", "D", "D_mini"]  # C tabled

# --- Track Registry ---
TRACK_CONFIG = {
    "A":      {"type": "dataframe", "path": DATA_DIR / "track_A_full.parquet"},
    "B":      {"type": "dataframe", "path": DATA_DIR / "track_B_curated.parquet"},
    "C":      {"type": "pipeline",  "path": DATA_DIR / "track_C_pipe.pkl"},       # tabled
    "D":      {"type": "pipeline",  "path": DATA_DIR / "track_D_pipe.pkl"},
    "D_mini": {"type": "pipeline",  "path": DATA_DIR / "track_D_mini_pipe.pkl"},
}

# --- Target Column Map ---
# maps (target, horizon) -> column name in targets.parquet
TARGET_COL_MAP = {
    (target, h): f"y_{target}_h{h}"
    for target in TARGET_LIST
    for h in HORIZON_LIST
}

print("Configuration set.")
print(f"  Window:  {WINDOW_MONTHS}m train / {TEST_MONTHS}m test / {STEP_MONTHS}m step")
print(f"  Targets: {TARGET_LIST}")
print(f"  Horizons: {HORIZON_LIST}")
print(f"  Active tracks: {ACTIVE_TRACKS}")

Configuration set.
  Window:  120m train / 1m test / 1m step
  Targets: ['INDPRO', 'PAYEMS', 'CPIAUCSL', 'SP500']
  Horizons: [1, 3, 6]
  Active tracks: ['A', 'B', 'D', 'D_mini']


In [9]:
# ============================================================
# Cell 3 — Configuration
# ============================================================

# --- Paths ---
DATA_DIR    = Path(REPO_ROOT) / "data" / "processed"
RESULTS_DIR = Path(REPO_ROOT) / "results"

# --- Window Parameters ---
WINDOW_MONTHS = 120   # training window length
TEST_MONTHS   = 1     # test window length
STEP_MONTHS   = 1     # fold step size

# --- Evaluation Parameters ---
HORIZON_LIST  = [1, 3, 6]
TARGET_LIST   = ["INDPRO", "PAYEMS", "CPIAUCSL", "S&P 500"]
ACTIVE_TRACKS = ["A", "B", "D", "D_mini"]  # C tabled

# --- Track Registry ---
TRACK_CONFIG = {
    "A":      {"type": "dataframe", "path": DATA_DIR / "track_A_full.parquet"},
    "B":      {"type": "dataframe", "path": DATA_DIR / "track_B_curated.parquet"},
    "C":      {"type": "pipeline",  "path": DATA_DIR / "track_C_pipe.pkl"},       # tabled
    "D":      {"type": "pipeline",  "path": DATA_DIR / "track_D_pipe.pkl"},
    "D_mini": {"type": "pipeline",  "path": DATA_DIR / "track_D_mini_pipe.pkl"},
}

# --- Target Column Map ---
# maps (target, horizon) -> column name in targets.parquet
TARGET_COL_MAP = {
    (target, h): f"y_{target}_h{h}"
    for target in TARGET_LIST
    for h in HORIZON_LIST
}

print("Configuration set.")
print(f"  Window:  {WINDOW_MONTHS}m train / {TEST_MONTHS}m test / {STEP_MONTHS}m step")
print(f"  Targets: {TARGET_LIST}")
print(f"  Horizons: {HORIZON_LIST}")
print(f"  Active tracks: {ACTIVE_TRACKS}")

Configuration set.
  Window:  120m train / 1m test / 1m step
  Targets: ['INDPRO', 'PAYEMS', 'CPIAUCSL', 'S&P 500']
  Horizons: [1, 3, 6]
  Active tracks: ['A', 'B', 'D', 'D_mini']


In [10]:
# ============================================================
# Cell 4 — Data Loading
# ============================================================

# --- Load track DataFrames ---
track_data = {}

for track_name, config in TRACK_CONFIG.items():
    if track_name not in ACTIVE_TRACKS:
        print(f"  Track {track_name:<8} SKIPPED (inactive)")
        continue
    if config["type"] == "dataframe":
        track_data[track_name] = pd.read_parquet(config["path"])
        print(f"  Track {track_name:<8} loaded   shape={track_data[track_name].shape}")
    elif config["type"] == "pipeline":
        with open(config["path"], "rb") as f:
            track_data[track_name] = pickle.load(f)
        print(f"  Track {track_name:<8} loaded   pipeline={type(track_data[track_name]).__name__}")

# --- Load stationary panel (needed for pipeline tracks) ---
stationary_panel = pd.read_parquet(DATA_DIR / "stationary_panel.parquet")
print(f"\n  Stationary panel loaded   shape={stationary_panel.shape}")

# --- Load targets ---
targets = pd.read_parquet(DATA_DIR / "targets.parquet")
print(f"  Targets loaded            shape={targets.shape}")

# --- Load USREC ---
usrec = pd.read_parquet(DATA_DIR / "usrec.parquet")
print(f"  USREC loaded              shape={usrec.shape}")

print("\nData loading complete.")

  Track A        loaded   shape=(767, 123)
  Track B        loaded   shape=(767, 6)
  Track C        SKIPPED (inactive)
  Track D        loaded   pipeline=Pipeline
  Track D_mini   loaded   pipeline=Pipeline

  Stationary panel loaded   shape=(767, 123)
  Targets loaded            shape=(806, 20)
  USREC loaded              shape=(806, 1)

Data loading complete.


In [11]:
print("Track A start:", track_data["A"].index.min())
print("Track A end:  ", track_data["A"].index.max())
print("Targets start:", targets.index.min())
print("Targets end:  ", targets.index.max())

Track A start: 1962-04-01 00:00:00
Track A end:   2026-02-01 00:00:00
Targets start: 1959-01-01 00:00:00
Targets end:   2026-02-01 00:00:00


In [13]:
for name in ACTIVE_TRACKS:
    if isinstance(track_data[name], pd.DataFrame):
        print(f"  Track {name:<8} {track_data[name].index.min()} -> {track_data[name].index.max()}")
print(f"  Stationary   {stationary_panel.index.min()} -> {stationary_panel.index.max()}")

  Track A        1962-04-01 00:00:00 -> 2026-02-01 00:00:00
  Track B        1962-04-01 00:00:00 -> 2026-02-01 00:00:00
  Stationary   1962-04-01 00:00:00 -> 2026-02-01 00:00:00


In [14]:
# ============================================================
# Cell 5 — Fold Generator
# Fixed sliding window pseudo-OOS splits
# ============================================================

def generate_folds(index, window_months, test_months, step_months):
    """
    Generate fixed sliding window pseudo-OOS folds.

    Parameters
    ----------
    index         : pd.DatetimeIndex — full date index of the panel
    window_months : int — training window length in months
    test_months   : int — test window length in months
    step_months   : int — step size between folds in months

    Returns
    -------
    folds : list of dict with train/test date boundaries
    """
    folds = []
    n = len(index)

    for start in range(0, n - window_months - test_months + 1, step_months):
        train_start = index[start]
        train_end   = index[start + window_months - 1]
        test_start  = index[start + window_months]
        test_end    = index[start + window_months + test_months - 1]

        folds.append({
            "fold_id":     len(folds),
            "train_start": train_start,
            "train_end":   train_end,
            "test_start":  test_start,
            "test_end":    test_end,
        })

    return folds


# generate folds using track A index as reference
folds = generate_folds(
    index         = track_data["A"].index,
    window_months = WINDOW_MONTHS,
    test_months   = TEST_MONTHS,
    step_months   = STEP_MONTHS,
)

print(f"  Total folds: {len(folds)}")
print(f"\n  First fold:")
print(f"    train: {folds[0]['train_start'].date()} -> {folds[0]['train_end'].date()}")
print(f"    test:  {folds[0]['test_start'].date()}  -> {folds[0]['test_end'].date()}")
print(f"\n  Last fold:")
print(f"    train: {folds[-1]['train_start'].date()} -> {folds[-1]['train_end'].date()}")
print(f"    test:  {folds[-1]['test_start'].date()}  -> {folds[-1]['test_end'].date()}")

  Total folds: 647

  First fold:
    train: 1962-04-01 -> 1972-03-01
    test:  1972-04-01  -> 1972-04-01

  Last fold:
    train: 2016-02-01 -> 2026-01-01
    test:  2026-02-01  -> 2026-02-01


In [15]:
# ============================================================
# Cell 6 — Metric Functions
# ============================================================

def compute_naive_mae(y_train):
    """
    Lag-1 random walk MAE on training target.
    Used as denominator for MASE and RMSSE.
    """
    y = np.array(y_train)
    return np.mean(np.abs(y[1:] - y[:-1]))


def compute_metrics(y_true, y_pred, y_train):
    """
    Compute all evaluation metrics.

    Parameters
    ----------
    y_true  : array-like — actual target values from test window
    y_pred  : array-like — model predictions
    y_train : array-like — training target values (for scaling denominator)

    Returns
    -------
    dict of metric name -> float
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mae       = mean_absolute_error(y_true, y_pred)
    mse       = mean_squared_error(y_true, y_pred)
    median_ae = median_absolute_error(y_true, y_pred)

    naive_mae = compute_naive_mae(y_train)

    # avoid division by zero on flat training series
    if naive_mae == 0:
        mase  = np.nan
        rmsse = np.nan
    else:
        mase  = mae / naive_mae
        rmsse = np.sqrt(mse) / naive_mae

    return {
        "mse":       mse,
        "mae":       mae,
        "median_ae": median_ae,
        "mase":      mase,
        "rmsse":     rmsse,
    }

print("Metric functions defined.")

Metric functions defined.


In [16]:
# ============================================================
# Cell 7 — Model Registry
# ============================================================

# --- Dummy model for loop validation ---
def dummy_model(train_df, test_df, target_col, horizon):
    """
    Returns random predictions with correct output shape.
    Used to validate loop mechanics before real models are integrated.
    """
    preds = np.random.randn(len(test_df))
    dates = test_df.index
    return preds, dates


# --- Registry ---
# key   : model name (used in results table)
# value : callable with signature (train_df, test_df, target_col, horizon)
#         -> (np.array of predictions, pd.DatetimeIndex of dates)

MODEL_REGISTRY = {
    "dummy": dummy_model,
    # "sigsvr":          sigsvr_model,    # David
    # "qrc":             qrc_model,       # William
    # "arima":           arima_model,     # Jack
    # "konduri_baseline": konduri_model,  # Manuel
}

print(f"Models registered: {list(MODEL_REGISTRY.keys())}")

Models registered: ['dummy']


In [17]:
# ============================================================
# Cell 8 — Pipeline Helper
# Fit/transform pipeline tracks before passing to models
# ============================================================

def prepare_track(track_name, fold):
    """
    Slice and prepare train/test DataFrames for a given track and fold.

    For DataFrame tracks (A, B): slice directly by date.
    For Pipeline tracks (D, D_mini): slice stationary panel by date,
        fit pipeline on train only, transform both, return DataFrames.

    Parameters
    ----------
    track_name : str — one of ACTIVE_TRACKS
    fold       : dict — fold definition with train/test date boundaries

    Returns
    -------
    train_df : pd.DataFrame
    test_df  : pd.DataFrame
    """
    config = TRACK_CONFIG[track_name]

    if config["type"] == "dataframe":
        df = track_data[track_name]
        train_df = df.loc[fold["train_start"]:fold["train_end"]]
        test_df  = df.loc[fold["test_start"]:fold["test_end"]]

    elif config["type"] == "pipeline":
        # slice stationary panel
        train_raw = stationary_panel.loc[fold["train_start"]:fold["train_end"]]
        test_raw  = stationary_panel.loc[fold["test_start"]:fold["test_end"]]

        # fit on train only — no leakage
        pipeline = pickle.loads(pickle.dumps(track_data[track_name]))  # fresh copy
        pipeline.fit(train_raw)

        # transform both
        train_transformed = pipeline.transform(train_raw)
        test_transformed  = pipeline.transform(test_raw)

        # return as DataFrames preserving date index
        n_cols = train_transformed.shape[1]
        cols   = [f"f{i}" for i in range(n_cols)]

        train_df = pd.DataFrame(train_transformed, index=train_raw.index, columns=cols)
        test_df  = pd.DataFrame(test_transformed,  index=test_raw.index,  columns=cols)

    return train_df, test_df

print("Pipeline helper defined.")

Pipeline helper defined.


In [18]:
# ============================================================
# Cell 9 — Evaluation Loop
# ============================================================

results = []

total_runs = len(folds) * len(ACTIVE_TRACKS) * len(TARGET_LIST) * len(HORIZON_LIST) * len(MODEL_REGISTRY)
completed  = 0

for fold in folds:
    for track_name in ACTIVE_TRACKS:

        # slice and prepare track data once per fold/track
        train_df, test_df = prepare_track(track_name, fold)

        for target in TARGET_LIST:
            for horizon in HORIZON_LIST:

                target_col = TARGET_COL_MAP[(target, horizon)]

                # slice target series
                y_train = targets.loc[train_df.index, target_col]
                y_test  = targets.loc[test_df.index,  target_col]

                # NaN guard — skip if target unavailable
                if y_test.isna().any() or y_train.isna().any():
                    continue

                for model_name, model_fn in MODEL_REGISTRY.items():

                    row = {
                        "model":       model_name,
                        "track":       track_name,
                        "target":      target,
                        "horizon":     horizon,
                        "fold_id":     fold["fold_id"],
                        "train_start": fold["train_start"],
                        "train_end":   fold["train_end"],
                        "test_start":  fold["test_start"],
                        "test_end":    fold["test_end"],
                        "error":       None,
                    }

                    try:
                        t0 = time.time()
                        y_pred, dates = model_fn(train_df, test_df, target_col, horizon)
                        row["runtime_sec"] = round(time.time() - t0, 4)

                        metrics = compute_metrics(y_test.values, y_pred, y_train.values)
                        row.update(metrics)

                    except Exception as e:
                        row["runtime_sec"] = np.nan
                        row["mse"]         = np.nan
                        row["mae"]         = np.nan
                        row["median_ae"]   = np.nan
                        row["mase"]        = np.nan
                        row["rmsse"]       = np.nan
                        row["error"]       = str(e)

                    results.append(row)
                    completed += 1

                    if completed % 1000 == 0:
                        print(f"  {completed}/{total_runs} runs complete")

print(f"\nLoop complete. Total rows: {len(results)}")

  1000/31056 runs complete
  2000/31056 runs complete
  3000/31056 runs complete
  4000/31056 runs complete
  5000/31056 runs complete
  6000/31056 runs complete
  7000/31056 runs complete
  8000/31056 runs complete
  9000/31056 runs complete
  10000/31056 runs complete
  11000/31056 runs complete
  12000/31056 runs complete
  13000/31056 runs complete
  14000/31056 runs complete
  15000/31056 runs complete
  16000/31056 runs complete
  17000/31056 runs complete
  18000/31056 runs complete
  19000/31056 runs complete
  20000/31056 runs complete
  21000/31056 runs complete
  22000/31056 runs complete
  23000/31056 runs complete
  24000/31056 runs complete
  25000/31056 runs complete
  26000/31056 runs complete
  27000/31056 runs complete
  28000/31056 runs complete
  29000/31056 runs complete
  30000/31056 runs complete

Loop complete. Total rows: 30896


In [19]:
# ============================================================
# Cell 10 — Save Results
# ============================================================

results_df = pd.DataFrame(results)

# --- Save per model ---
for model_name in results_df["model"].unique():
    model_df = results_df[results_df["model"] == model_name]
    out_path = RESULTS_DIR / f"{model_name}_raw.parquet"
    model_df.to_parquet(out_path, index=False)
    print(f"  Saved: {out_path.name}   rows={len(model_df)}")

# --- Summary table ---
summary = (
    results_df
    .groupby(["model", "track", "target", "horizon"])
    [["mse", "mae", "median_ae", "mase", "rmsse"]]
    .mean()
    .round(6)
)

print("\nSummary — mean metrics across folds:")
print(summary.to_string())

  Saved: dummy_raw.parquet   rows=30896

Summary — mean metrics across folds:
                                    mse       mae  median_ae        mase       rmsse
model track  target   horizon                                                       
dummy A      CPIAUCSL 1        0.997239  0.792759   0.792759  428.893693  428.893693
                      3        1.103167  0.824882   0.824882  365.548719  365.548719
                      6        0.999978  0.795126   0.795126  353.651392  353.651392
             INDPRO   1        0.985872  0.783254   0.783254  126.594177  126.594177
                      3        1.003373  0.801344   0.801344  126.682524  126.682524
                      6        1.043335  0.819985   0.819985  119.450689  119.450689
             PAYEMS   1        1.073036  0.823676   0.823676  693.714840  693.714840
                      3        1.048409  0.825163   0.825163  624.068421  624.068421
                      6        1.030158  0.803216   0.803216  531.657230

In [20]:
# ============================================================
# Cell 11 — USREC Disaggregation
# Join recession indicator for recession vs expansion analysis
# ============================================================

# rename for clean merge
usrec_col = usrec.columns[0]
usrec_clean = usrec.rename(columns={usrec_col: "usrec"})

# join on test_start date
results_df = results_df.merge(
    usrec_clean,
    left_on="test_start",
    right_index=True,
    how="left"
)

# disaggregated summary
summary_disagg = (
    results_df
    .groupby(["model", "track", "target", "horizon", "usrec"])
    [["mse", "mae", "median_ae", "mase", "rmsse"]]
    .mean()
    .round(6)
)

print("Disaggregated summary — recession (1) vs expansion (0):")
print(summary_disagg.to_string())

Disaggregated summary — recession (1) vs expansion (0):
                                          mse       mae  median_ae        mase       rmsse
model track  target   horizon usrec                                                       
dummy A      CPIAUCSL 1       0      1.022449  0.802762   0.802762  438.377179  438.377179
                              1      0.802369  0.715439   0.715439  355.588907  355.588907
                      3       0      1.088438  0.817553   0.817553  364.853613  364.853613
                              1      1.216621  0.881335   0.881335  370.902916  370.902916
                      6       0      1.049655  0.813941   0.813941  364.261434  364.261434
                              1      0.619340  0.650963   0.650963  272.355533  272.355533
             INDPRO   1       0      0.986016  0.784187   0.784187  127.069226  127.069226
                              1      0.984756  0.776042   0.776042  122.922173  122.922173
                      3       0   